In [2]:
import anthropic
import pathlib
import csv
import pandas as pd
from dotenv import load_dotenv
import re

In [3]:
load_dotenv("../etc/.env")  # loads .env into environment variables

True

In [4]:
client = anthropic.Anthropic()

In [6]:
df = pd.read_csv("exercises_reactividad_organica.csv")

In [7]:
df.head()

,subject,year,topic,exam,exercise,statement
0,química,2000,Reactividad Orgánica,Junio,Ejercicio 4 Opción B,Complete y ajuste las siguientes reacciones or...
1,química,2000,Reactividad Orgánica,Reserva 2,Ejercicio 4 Opción B,"sustitución) a que corresponden.\na) CH,CH = C..."
2,química,2000,Reactividad Orgánica,Reserva 3,Ejercicio 4 Opción A,Complete las siguientes reacciones y ajuste la...
3,química,2000,Reactividad Orgánica,Reserva 4,Ejercicio 4 Opción B,Defina serie homologa e indique cuáles de los ...
4,quimica,2000,Reactividad Orgánica,Septiembre,Ejercicio 4 Opción B,"Dados los siguientes compuestos orgánicos: CH,..."


In [8]:
df.statement = df.statement.str.replace("\n", " | ")

In [9]:
df_chunks = [df[i:i+50] for i in range(0, len(df), 50) ]

In [ ]:
# dividir el dataframe en varios trozos

(143, 6)

In [10]:
def clasificar_ejercicios(dataset: pd.DataFrame) -> list[str]:
    subject = dataset.subject.unique()[0]   # dataset, no df
    unique_topics = dataset.topic.unique()

    for topic in unique_topics:
        system_prompt = pathlib.Path(f"{topic}.md").read_text()

        subset = dataset[dataset.topic == topic]        # iloc no hace falta aquí
        data_csv = subset.to_csv("data_csv.csv", index=False)
        
        data_text = pathlib.Path("data_csv.csv").read_text()        # subset, no df completo

        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=8192,
            system=system_prompt,
            messages=[{"role": "user",
                       "content": f"Here is the data: \n\n{data_text}"
                       }]
        )

    return response.content[0].text.strip().splitlines()

La respuesta incluye los enunciados, que dan problemas de parseo al convertir a csv. Elimino los enunciados.

In [11]:
def remove_statement(line):
    start, end = [i.start() for i in re.finditer('"', line)]
    line = line[:start-1] + line[end+1:] 
    return line

In [41]:
response = clasificar_ejercicios(df_chunks[2])

In [42]:
response[:10]

['# Análisis y clasificación completada',
 '',
 'He analizado cada ejercicio según los criterios especificados. Aquí está el archivo CSV actualizado con la columna "tipo_ejercicio":',
 '',
 '```csv',
 'subject,year,topic,exam,exercise,statement,tipo_ejercicio',
 'quimica,2018,Reactividad Orgánica,Reserva 3,Ejercicio 4 Opción B,"Empleando compuestos de 4 átomos de carbono, represente: | a) Dos hidrocarburos que sean isómeros de cadena entre sí. | b) Dos hidrocarburos que sean isómeros cis-trans. | c) Un alcohol que desvíe el plano de la luz polarizada. . | QUIMICA. 2018. RESERVA 3. EJERCICIO 4. OPCIÓN B",Isomería;Hidrocarburos',
 'quimica,2018,Reactividad Orgánica,Reserva 4,Ejercicio 4 Opción B,"Escriba las fórmulas de los siguientes compuestos: | a) El aldehído que es isómero del propen-2-o1 (CH , = COH-CH;). | b) Un alqueno de 4 átomos de carbono que no presente isomería cis-trans. | €) Un compuesto con dos carbonos quirales. . | QUIMICA. 2018. RESERVA 4. EJERCICIO 4. OPCIÓN B",Isomer

In [43]:
response = response[6:]

In [44]:
response[:5]

['quimica,2018,Reactividad Orgánica,Reserva 3,Ejercicio 4 Opción B,"Empleando compuestos de 4 átomos de carbono, represente: | a) Dos hidrocarburos que sean isómeros de cadena entre sí. | b) Dos hidrocarburos que sean isómeros cis-trans. | c) Un alcohol que desvíe el plano de la luz polarizada. . | QUIMICA. 2018. RESERVA 3. EJERCICIO 4. OPCIÓN B",Isomería;Hidrocarburos',
 'quimica,2018,Reactividad Orgánica,Reserva 4,Ejercicio 4 Opción B,"Escriba las fórmulas de los siguientes compuestos: | a) El aldehído que es isómero del propen-2-o1 (CH , = COH-CH;). | b) Un alqueno de 4 átomos de carbono que no presente isomería cis-trans. | €) Un compuesto con dos carbonos quirales. . | QUIMICA. 2018. RESERVA 4. EJERCICIO 4. OPCIÓN B",Isomería;Hidrocarburos',
 'quimica,2018,Reactividad Orgánica,Septiembre,Ejercicio 4 Opción A,"Dados los siguientes reactivos HI,I ,, H ,/catalizador, NaOH y H ,O/H ,SO ,, ¿cuál de ellos | sería el adecuado para obtener CH , —-CH,-—CH,-CH(OH) | reacción correspondiente

In [45]:
processed_response = []
# lista para guardar las líneas a las que no pude eliminar el statement
# contiene la cabecera (response[0])
failed_lines = []
failed_lines.append(response[0])

for line in response:
    try:
        processed_response.append( remove_statement(line) )
    except ValueError:
        failed_lines.append(line)

In [46]:
processed_response[:5]

['quimica,2018,Reactividad Orgánica,Reserva 3,Ejercicio 4 Opción B,Isomería;Hidrocarburos',
 'quimica,2018,Reactividad Orgánica,Reserva 4,Ejercicio 4 Opción B,Isomería;Hidrocarburos',
 'quimica,2018,Reactividad Orgánica,Septiembre,Ejercicio 4 Opción A,Reacciones orgánicas',
 'quimica,2018,Reactividad Orgánica,Septiembre,Ejercicio 4 Opción B,Isomería',
 'quimica,2019,Reactividad Orgánica,Junio,Ejercicio 4 Opción B,Isomería;Reacciones orgánicas']

In [47]:
failed_to_clean_data = "\n".join(failed_lines)

In [48]:
print(failed_to_clean_data)

quimica,2018,Reactividad Orgánica,Reserva 3,Ejercicio 4 Opción B,"Empleando compuestos de 4 átomos de carbono, represente: | a) Dos hidrocarburos que sean isómeros de cadena entre sí. | b) Dos hidrocarburos que sean isómeros cis-trans. | c) Un alcohol que desvíe el plano de la luz polarizada. . | QUIMICA. 2018. RESERVA 3. EJERCICIO 4. OPCIÓN B",Isomería;Hidrocarburos
química,2019,Reactividad Orgánica,Junio,Ejercicio 4 Opción A,Razone si son verdaderas o falsas las siguientes afirmaciones: | a) La regla de Markovnikov predice qué compuesto mayoritario se forma en las reacciones de | eliminación. | b) Un alquino puede adicionar halógenos. | c) Un compuesto que desvía el plano de la luz polarizada presenta isomería geométrica. | QUÍMICA. 2019. JUNIO. EJERCICIO 4. OPCIÓN A",Verdadero / falso;Isomería;Reacciones orgánicas
química,2019,Reactividad Orgánica,Reserva 4,Ejercicio 4 Opción A,"Las fórmulas moleculares de tres hidrocarburos lineales son: C ,H ,, C ,H, y C ,H,, . Razone | si son ver

In [49]:
classified_data = "\n".join(processed_response)

In [50]:
print(classified_data)

quimica,2018,Reactividad Orgánica,Reserva 3,Ejercicio 4 Opción B,Isomería;Hidrocarburos
quimica,2018,Reactividad Orgánica,Reserva 4,Ejercicio 4 Opción B,Isomería;Hidrocarburos
quimica,2018,Reactividad Orgánica,Septiembre,Ejercicio 4 Opción A,Reacciones orgánicas
quimica,2018,Reactividad Orgánica,Septiembre,Ejercicio 4 Opción B,Isomería
quimica,2019,Reactividad Orgánica,Junio,Ejercicio 4 Opción B,Isomería;Reacciones orgánicas
química,2019,Reactividad Orgánica,Reserva 1,Ejercicio 4 Opción A,Isomería;Reacciones orgánicas
química,2019,Reactividad Orgánica,Reserva 2,Ejercicio 4 Opción A,Isomería
química,2019,Reactividad Orgánica,Reserva 2,Ejercicio 4 Opción B,Reacciones orgánicas
quiímica,2019,Reactividad Orgánica,Reserva 3,Ejercicio 4 Opción B,Isomería
quiímica,2019,Reactividad Orgánica,Septiembre,Ejercicio 4 Opcion A,Reacciones orgánicas
química,2019,Reactividad Orgánica,Septiembre,Ejercicio 4 Opción B,Isomería
quimica,2020,Reactividad Orgánica,Junio,B6,Isomería;Reacciones orgánicas
quími

In [51]:
with open("failed_to_clean_data.txt", "w", encoding="utf-8") as file:
    file.writelines(failed_to_clean_data)

In [52]:
with open("classified_reactividad_organica.csv", "a", encoding="utf-8") as file:
    file.writelines(classified_data)

In [54]:
df2 = pd.read_csv("classified_reactividad_organica.csv", encoding="utf-8")

In [57]:
df2.subject = "química"

In [60]:
df2.sample(10)

,subject,year,topic,exam,exercise,tipo_ejercicio
84,química,2017,Reactividad Orgánica,Reserva 1,Ejercicio 4 Opción B,Reacciones orgánicas | Hidrocarburos
34,química,2005,Reactividad Orgánica,Reserva 3,Ejercicio 4 Opcióon A,Reacciones orgánicas
43,química,2007,Reactividad Orgánica,Reserva 4,Ejercicio 4 Opción A,Reacciones orgánicas
71,química,2014,Reactividad Orgánica,Reserva 3,Ejercicio 4 Opción B,Reacciones orgánicas
27,química,2004,Reactividad Orgánica,Junio,Ejercicio 4 Opción B,Isomería
39,química,2006,Reactividad Orgánica,Septiembre,Ejercicio 4 Opción B,Isomería
14,química,2001,Reactividad Orgánica,Reserva 2,Ejercicio 4 Opción B,Reacciones orgánicas
104,química,2019,Reactividad Orgánica,Junio,Ejercicio 4 Opción B,Isomería | Reacciones orgánicas
21,química,2002,Reactividad Orgánica,Reserva 4,Ejercicio 4 Opción A,Hidrocarburos
120,química,2021,Reactividad Orgánica,Reserva 4,Ejercicio B4,Reacciones orgánicas | Isomería


In [59]:
df2.tipo_ejercicio = df2.tipo_ejercicio.str.replace(";", " | ")

In [61]:
df2 = df2.sort_values(["year", "exam", "exercise"]).reset_index(drop = True)

In [62]:
df2.to_csv("classified_reactividad_organica.csv", index = False)